# 13 — Tabular Export

**Purpose:** Show every way to get TorchLens data into tabular form: `Trace.to_pandas()`,
per-record `to_pandas()` (Op/Layer), and the `torchlens.export` submodule
(`csv`, `json`, `parquet`).  Show schema, column counts, and a read-back round-trip.

**Surfaces covered:**
- [ ] `torchlens.export` submodule — confirmed exists at `torchlens.export`
- [ ] `torchlens.export.csv(trace, path)` — write CSV, read back with pandas
- [ ] `torchlens.export.json(trace, path)` — write JSON (records orient), read back
- [ ] `torchlens.export.parquet(trace, path)` — write Parquet, read back
- [ ] `Trace.to_pandas()` — full trace as multi-row DataFrame (one row = one Op)
- [ ] `Layer.to_pandas()` — layer-aggregated single-row DataFrame
- [ ] `Op.to_pandas()` — pass-level single-row DataFrame (more columns than Layer)
- [ ] Schema comparison: column counts across Trace/Layer/Op DataFrames
- [ ] Key columns: `layer_label`, `layer_type`, `activation_memory`
- [ ] Round-trip: write -> read back -> confirm shape and key values
- [ ] Profiling exports: `chrome_trace`, `speedscope`, `flamegraph`, `memory_timeline`
- [ ] Graph/report exports: `html`, `svg`, `netron`, `model_explorer`
- [ ] Science export: `xarray` (Dataset: `out` is the presentation x neuroid DataArray; each buffer is its own module-state variable)
- [ ] Tracker exports (guarded): `tensorboard`, `wandb`, `mlflow`, `aim`; `chrome_trace_diff`

## 1. Environment setup

In [ ]:
import json as _json
import os
import pathlib
import sys
import tempfile

# Pin imports to THIS checkout: the notebook dir (for _models.py) plus the repo
# root, so `import torchlens` audits the code this notebook ships with -- not a
# pip-installed copy from another checkout.
_NB_DIR = pathlib.Path.cwd()
if not (_NB_DIR / "_models.py").exists():
    _NB_DIR = next(
        p
        for p in [_NB_DIR / "notebooks" / "audit", *_NB_DIR.parents]
        if (p / "_models.py").exists()
    )
_REPO_ROOT = _NB_DIR.parents[1]
sys.path.insert(0, str(_NB_DIR))
sys.path.insert(0, str(_REPO_ROOT))

import warnings as _w

_w.filterwarnings("ignore", message=".*pynvml.*")  # torch/cuda probe noise
import torch

import torchlens as tl

assert pathlib.Path(tl.__file__).is_relative_to(_REPO_ROOT), (
    f"torchlens imported from {tl.__file__} -- expected this checkout"
)
from _models import ZOO

print(f"torchlens version : {tl.__version__}")
print(f"torch version     : {torch.__version__}")

TMPDIR = tempfile.mkdtemp(prefix="tl_audit_13_")
print(f"temp dir          : {TMPDIR}")

## 2. Verify `torchlens.export` module path

In [ ]:
# Verify the module exists and locate it
try:
    import torchlens.export as tl_export

    print("torchlens.export EXISTS")
    print("module file:", tl_export.__file__)
    print()
    # Show public names
    public = (
        tl_export.__all__
        if hasattr(tl_export, "__all__")
        else [n for n in dir(tl_export) if not n.startswith("_")]
    )
    print("Public names:", sorted(public))
except ImportError as e:
    print(f"\u26a0\ufe0f  GAP: torchlens.export NOT importable: {e}")

In [ ]:
import inspect

# Confirm csv/json/parquet are callable with correct signatures
for name in ["csv", "json", "parquet"]:
    fn = getattr(tl_export, name, None)
    if fn is None:
        print(f"\u26a0\ufe0f  GAP: torchlens.export.{name} not found")
    else:
        sig = inspect.signature(fn)
        print(f"torchlens.export.{name}{sig}")

## 3. Capture a trace for all export demos

In [ ]:
model, x = ZOO["tiny_mlp"]()
trace = tl.trace(model, x)

print(repr(trace))
print("layer_labels:", trace.layer_labels)

## 4. `Trace.to_pandas()` — full trace DataFrame

In [ ]:
trace_df = trace.to_pandas()

print("trace.to_pandas() shape:", trace_df.shape)
print(f"  {trace_df.shape[0]} rows (one per Op)  x  {trace_df.shape[1]} columns")
print()
print("All columns:")
# Print in groups of 10 for readability
cols = list(trace_df.columns)
for i in range(0, len(cols), 10):
    print(" ", cols[i : i + 10])

In [ ]:
# Show key columns
key_cols = [
    c
    for c in ["layer_label", "layer_type", "activation_memory", "label", "func", "grad_fn_name"]
    if c in trace_df.columns
]
print("Key column view:")
print(trace_df[key_cols].to_string())

## 5. `Layer.to_pandas()` — layer-aggregated view

In [ ]:
layer = trace.layers["relu_1_2"]
layer_df = layer.to_pandas()

print("Layer.to_pandas() shape:", layer_df.shape)
print(f"  1 row (this layer)  x  {layer_df.shape[1]} columns")
print()
# Show first 12 columns
print("First 12 columns:")
print(layer_df.iloc[:, :12].to_string())

## 6. `Op.to_pandas()` — pass-level view

In [ ]:
op = trace.ops["relu_1_2:1"]
op_df = op.to_pandas()

print("Op.to_pandas() shape:", op_df.shape)
print(f"  1 row (this op/pass)  x  {op_df.shape[1]} columns")
print()
# Show first 12 columns
print("First 12 columns:")
print(op_df.iloc[:, :12].to_string())

## 7. Schema comparison — column counts across DataFrame levels

In [ ]:
print("=== DataFrame schema comparison ===")
print(f"  Trace.to_pandas()    : {len(trace_df.columns):4d} columns  ({trace_df.shape[0]} rows)")
print(f"  Layer.to_pandas()    : {len(layer_df.columns):4d} columns  (1 row per layer)")
print(f"  Op.to_pandas()       : {len(op_df.columns):4d} columns  (1 row per pass)")
print()
print("Note: Trace has a DIFFERENT column set from both Layer and Op.")
print("  Columns in Trace but not Op :", len(set(trace_df.columns) - set(op_df.columns)))
print("  Columns in Op but not Trace  :", len(set(op_df.columns) - set(trace_df.columns)))
print("  Shared columns               :", len(set(trace_df.columns) & set(op_df.columns)))

## 8. `torchlens.export.csv` — write CSV and read back

In [ ]:
csv_path = os.path.join(TMPDIR, "trace.csv")
written_path = tl_export.csv(trace, csv_path)

print(f"csv() wrote to: {written_path}")
print(f"file size: {os.path.getsize(csv_path):,} B")

In [ ]:
# Read back with pandas
import pandas as pd

# The export leads with a "# torchlens capture honesty: ..." comment line
# (status + verification facts ride every tabular artifact now); tell pandas:
csv_df = pd.read_csv(csv_path, comment="#")
print("Round-trip CSV shape:", csv_df.shape)
print()

# Verify key column values survived
key_cols = [c for c in ["layer_label", "layer_type", "activation_memory"] if c in csv_df.columns]
print("Key column round-trip:")
print(csv_df[key_cols].to_string())
print()
print("layer_label matches original:", list(csv_df["layer_label"]) == trace.layer_labels)

## 9. `torchlens.export.json` — write JSON and read back

In [ ]:
json_path = os.path.join(TMPDIR, "trace.json")
tl_export.json(trace, json_path)

print(f"json() wrote to: {json_path}")
print(f"file size: {os.path.getsize(json_path):,} B")

with open(json_path) as f:
    json_data = _json.load(f)

# The export is now a torchlens.table_export.v1 ENVELOPE: schema id,
# capture-honesty facts, orient, and the records under "rows".
print(f"JSON type: {type(json_data).__name__}")
print("schema        :", json_data["schema"])
print("capture status:", json_data["capture_honesty"]["capture_status"])
json_data = json_data["rows"]
print(f"Records: {len(json_data)}  (one per Op)")
print()
# Show first record's keys
if json_data:
    first_keys = list(json_data[0].keys())
    print(f"First record has {len(first_keys)} fields:")
    print(" ", first_keys[:10], "...")

In [ ]:
# Read back with pandas via orient='records'
with open(json_path) as f:
    _envelope = _json.load(f)
json_df = pd.DataFrame(_envelope["rows"])
print("Round-trip JSON shape:", json_df.shape)
key_cols = [c for c in ["layer_label", "layer_type", "activation_memory"] if c in json_df.columns]
print(json_df[key_cols].to_string())

## 10. `torchlens.export.parquet` — write Parquet and read back

In [ ]:
parquet_path = os.path.join(TMPDIR, "trace.parquet")

try:
    tl_export.parquet(trace, parquet_path)
    print(f"parquet() wrote to: {parquet_path}")
    print(f"file size: {os.path.getsize(parquet_path):,} B")
except ImportError as e:
    print(f"\u26a0\ufe0f  parquet requires pyarrow/fastparquet: {e}")
    parquet_path = None

In [ ]:
if parquet_path and os.path.exists(parquet_path):
    parquet_df = pd.read_parquet(parquet_path)
    print("Round-trip Parquet shape:", parquet_df.shape)
    key_cols = [
        c for c in ["layer_label", "layer_type", "activation_memory"] if c in parquet_df.columns
    ]
    print(parquet_df[key_cols].to_string())
    print()
    print("layer_label matches original:", list(parquet_df["layer_label"]) == trace.layer_labels)

## 11. Demo on a more complex model (demo_model)

In [ ]:
demo_model, demo_x = ZOO["demo_model"]()
demo_trace = tl.trace(demo_model, demo_x)

print(repr(demo_trace))
demo_df = demo_trace.to_pandas()
print(f"demo trace.to_pandas() shape: {demo_df.shape}")
print()

key_cols = [c for c in ["layer_label", "layer_type", "activation_memory"] if c in demo_df.columns]
print(demo_df[key_cols].to_string())

## 12. The full `torchlens.export` roster — every format exercised

`torchlens.export` ships 17 formats. Twelve run with no extra dependencies — each is
executed below and its artifact inspected. The tracker-integration formats
(`tensorboard`, `wandb`, `mlflow`, `aim`) take a live writer/run object and are
enumerated with their signatures; `chrome_trace_diff` needs two traces.

In [ ]:
# Dep-free formats: execute each and confirm the artifact
dep_free = [
    "chrome_trace",  # Chrome tracing JSON (chrome://tracing, Perfetto)
    "speedscope",  # speedscope.app profile
    "flamegraph",  # collapsed-stack text for flamegraph.pl
    "memory_timeline",  # per-op memory timeline
    "html",  # self-contained HTML report
    "svg",  # standalone SVG graph
    "netron",  # Netron-openable graph file
    "model_explorer",  # Google Model Explorer JSON
    "xarray",  # Dataset (no path argument)
]

for name in dep_free:
    fn = getattr(tl_export, name)
    try:
        if name == "xarray":
            result = fn(trace)
            print(f"{name:16s} -> {type(result).__name__} dims={dict(result.sizes)}")
        else:
            out = fn(trace, os.path.join(TMPDIR, f"fmt_{name}"))
            size = os.path.getsize(out)
            print(f"{name:16s} -> {out}  ({size:,} B)")
    except Exception as exc:
        print(f"⚠️ {name:14s} -> {type(exc).__name__}: {exc}")

In [ ]:
# Peek inside two of them so the artifact shape is visible, not just the file size
import json as _json2

with open(os.path.join(TMPDIR, "fmt_chrome_trace")) as f:
    ct = _json2.load(f)
events = ct if isinstance(ct, list) else ct.get("traceEvents", [])
print(f"chrome_trace: {len(events)} events; first:")
print("  ", {k: events[0][k] for k in list(events[0])[:6]})
print()

with open(os.path.join(TMPDIR, "fmt_flamegraph")) as f:
    fg_lines = f.read().splitlines()
print(f"flamegraph: {len(fg_lines)} collapsed-stack lines; first two:")
for ln in fg_lines[:2]:
    print("  ", ln[:100])

In [ ]:
# Tracker-integration formats: signature enumeration (each takes a live writer/run
# object, not a path -- exercising them would require the tracker service).
for name in ["tensorboard", "wandb", "mlflow", "aim", "chrome_trace_diff"]:
    fn = getattr(tl_export, name)
    try:
        sig = str(inspect.signature(fn))
        print(f"{name}{sig[:95]}")
    except (ValueError, TypeError):
        print(f"{name}  (not inspectable)")
print()

# wandb documents its extra properly:
try:
    tl_export.wandb(trace, None)
except Exception as exc:
    print(f"wandb without the extra -> {type(exc).__name__}: {exc}")
print()

# The path-instead-of-writer footgun, exercised (see GAP cell):
try:
    tl_export.tensorboard(trace, "/tmp/tb_out", step=0)
except Exception as exc:
    print(f"tensorboard(trace, '/tmp/tb_out') -> {type(exc).__name__}: {exc}")
    print("⚠️ GAP: a path -- the shape every sibling format takes -- fails with an")
    print("   opaque AttributeError instead of a type check naming SummaryWriter.")

---

## ⚠️ GAPs / ergonomic smells

- *(fixed since 2026-06)* **`export` is now in `tl.__all__`** — `tl.export.csv(...)`
  works without a separate import (this notebook keeps the explicit import for clarity).

- **Three DataFrame schemas, none a superset of another.** `Trace.to_pandas()`,
  `Layer.to_pandas()`, `Op.to_pandas()` have overlapping but distinct column sets.
  Joins on `layer_label` across them hit silent column conflicts; no shared schema
  constant is exposed.

- **`forward_flops` and `out_shape` are not `Trace.to_pandas()` columns** despite being
  natural asks; users must collect them from Op records.

- **Tracker exports (`tensorboard`/`wandb`/`mlflow`/`aim`) take a live writer/run object**
  — passing a path (the natural first guess, since every sibling takes a path) fails
  with an opaque `AttributeError: 'str' object has no attribute 'add_scalar'` instead of
  a type check naming the expected object.

- **`parquet` requires a third-party engine** (`pyarrow`/`fastparquet`); a plain
  `ImportError` surfaces if absent. `wandb`, by contrast, raises a clear
  "install torchlens[wandb]" message — the good pattern the others should follow.

- **`xarray` export requires all outs to share one presentation count** — models whose
  saved ops mix batch shapes raise `ValueError: All exported outs must share the same
  presentation count`; fine as a contract, but the error does not say which op violated
  it.